# Study and prepare missing text metadata

We examine `track_name`, `album_name`, and `record_label`, then replace missing values with the literal string `unknown` in copies of the training and test data. We preserve all rows, non-missing values, and raw CSV files.

The cleaned frames remain in memory as `train_clean` and `test_clean`. We record the findings and decision in `reports/text_metadata/summary.md`; this notebook does not overwrite the genre-only CSV in `data/processed/`.

## 1. Load the original datasets

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

DATA_DIR = next(
    (directory for directory in (
        Path.cwd() / "data/raw",
        Path.cwd().parent / "data/raw",
        Path.cwd(),
    ) if all((directory / name).is_file() for name in ("train.csv", "test.csv"))),
    None,
)
if DATA_DIR is None:
    raise FileNotFoundError("Set DATA_DIR to the folder containing train.csv and test.csv.")
train_raw = pd.read_csv(DATA_DIR / "train.csv")
test_raw = pd.read_csv(DATA_DIR / "test.csv")


## 2. Study the three columns

We count missing fields, their percentage, and the distinct non-missing values. Empty and whitespace-only strings are treated as missing alongside null values.

In [2]:
TEXT_COLUMNS = ("track_name", "album_name", "record_label")
MISSING_TEXT_VALUE = "unknown"


def missing_text_mask(series):
    return series.isna() | series.astype("string").str.strip().eq("").fillna(False)


def summarize_text_metadata(datasets):
    rows = []
    for name, frame in datasets.items():
        for column in TEXT_COLUMNS:
            missing = missing_text_mask(frame[column])
            rows.append({
                "dataset": name,
                "column": column,
                "missing values": int(missing.sum()),
                "% missing": round(100 * missing.mean(), 4),
                "distinct non-missing values": frame.loc[~missing, column].nunique(),
            })
    return pd.DataFrame(rows)


def fill_missing_text(frame):
    cleaned = frame.copy()
    for column in TEXT_COLUMNS:
        cleaned.loc[missing_text_mask(cleaned[column]), column] = MISSING_TEXT_VALUE
    return cleaned

missing_text_summary = summarize_text_metadata({"train": train_raw, "test": test_raw})
display(missing_text_summary)


,dataset,column,missing values,% missing,distinct non-missing values
0,train,track_name,4,0.0011,248383
1,train,album_name,1,0.0003,57310
2,train,record_label,38,0.0109,31558
3,test,track_name,1,0.0011,73246
4,test,album_name,0,0.0000,32169
5,test,record_label,14,0.0160,17780


## 3. Inspect affected tracks

We inspect a small sample of the rows with missing text. A missing name or label does not imply that the remaining features are unusable.

In [3]:
for name, frame in (("train", train_raw), ("test", test_raw)):
    affected = pd.concat([missing_text_mask(frame[column]) for column in TEXT_COLUMNS], axis=1).any(axis=1)
    print(f"{name}: {int(affected.sum()):,} tracks with at least one missing text field")
    display(frame.loc[affected, ["id", *TEXT_COLUMNS]].head(5))


train: 42 tracks with at least one missing text field
test: 15 tracks with at least one missing text field


,id,track_name,album_name,record_label
31990,31991,Last Night,Od Nashir,NaN
31991,31992,Fall On Me,Od Nashir,NaN
31992,31993,Mocher Gar'Inim,Od Nashir,NaN
31993,31994,Ani Ma'Amin,Od Nashir,NaN
31994,31995,Hallelukah,Od Nashir,NaN


,id,track_name,album_name,record_label
8105,358157,Tziyon Medley,Od Nashir,NaN
8106,358158,I'M A Train,Od Nashir,NaN
8107,358159,Stand By Me,Od Nashir,NaN
8108,358160,America The Beautiful,Od Nashir,NaN
29675,379727,Rudolf The Red Nosed Reindeer,Matt & Mike'S Brown Christmas,NaN


## 4. Fill missing values and check the result

We use `unknown` consistently in both datasets and retain the remaining text exactly as provided. We keep every test row so that each track can receive a prediction. The missing-text rule is independent of genre encoding and does not determine whether we use these names as model features.

In [4]:
train_clean = fill_missing_text(train_raw)
test_clean = fill_missing_text(test_raw)

for original, cleaned in ((train_raw, train_clean), (test_raw, test_clean)):
    pd.testing.assert_frame_equal(
        original.drop(columns=list(TEXT_COLUMNS)),
        cleaned.drop(columns=list(TEXT_COLUMNS)),
    )
    for column in TEXT_COLUMNS:
        missing = missing_text_mask(original[column])
        assert not missing_text_mask(cleaned[column]).any()
        assert cleaned.loc[missing, column].eq(MISSING_TEXT_VALUE).all()
        pd.testing.assert_series_equal(original.loc[~missing, column], cleaned.loc[~missing, column])
print("Filled missing text values; all rows and non-missing values were preserved.")

display(summarize_text_metadata({"train_clean": train_clean, "test_clean": test_clean}))

Filled missing text values; all rows and non-missing values were preserved.


,dataset,column,missing values,% missing,distinct non-missing values
0,train_clean,track_name,0,0.0,248384
1,train_clean,album_name,0,0.0,57311
2,train_clean,record_label,0,0.0,31559
3,test_clean,track_name,0,0.0,73247
4,test_clean,album_name,0,0.0,32169
5,test_clean,record_label,0,0.0,17781


## 5. Record our decision

In [5]:
report_dir = DATA_DIR.parent.parent / "reports/text_metadata"
report_dir.mkdir(parents=True, exist_ok=True)
lines = [
    "# Missing text metadata", "",
    "We studied missing values in `track_name`, `album_name`, and `record_label` in the original training and test data. "
    "We treated null values, empty strings, and whitespace-only fields as missing.", "",
    "| Dataset | Column | Missing values | Missing (%) |",
    "| --- | --- | ---: | ---: |",
]
for _, row in missing_text_summary.iterrows():
    lines.append(f"| {row['dataset']} | `{row['column']}` | {row['missing values']} | {row['% missing']:.4f} |")
lines += [
    "", "## Decision", "",
    "We replaced missing values in these three columns with `unknown` in copies of both datasets. "
    "We retained every track because the remaining metadata and audio features can still be used. "
    "We preserved non-missing text, including existing names such as `Unknown`, and left the raw CSV files unchanged. "
    "This preparation does not require target information or statistics learned from either dataset.", "",
    "Our working notebook keeps the cleaned copies in memory. We included the same transformation "
    "in `delivery/music_hit_prediction.ipynb`, before genre encoding. "
    "Whether to use these text columns as model features remains a separate modeling decision.", "",
]
(report_dir / "summary.md").write_text("\n".join(lines), encoding="utf-8")
print(f"Report saved to {report_dir / 'summary.md'}")


Report saved to /home/goncalotorcato/ist1109358/CD/Projeto/CD/reports/text_metadata/summary.md
